# OpenVLA-7B LoRA — BASELINE (Uniform LoRA) — LIBERO-Object (Kaggle T4)
### layer-selective-peft-vla project — Phase 1: real baseline run for the paper

This is **not** the smoke test. The smoke test (batch_size=1, max_steps=10) already passed and proved the pipeline works end-to-end. This notebook reuses that exact same proven pipeline unchanged, and only extends it into a real run long enough to produce a usable baseline result: **uniform LoRA** applied to every transformer layer, at a fixed LoRA parameter budget (`lora_rank=32`), trained for real on LIBERO-Object.

The MMD-selective LoRA condition (the actual novel comparison) is a **separate, not-yet-built** notebook — this one only produces the baseline half of the comparison.

**What changed vs. the smoke test:** only `MAX_STEPS`, `SAVE_STEPS`, and the run/checkpoint name, all defined in one config cell below. Everything else — installs, patches, dataset, model config — is byte-for-byte the same as what already passed.

## 1. Install pinned dependencies
(Unchanged from the proven smoke-test notebook.)

In [1]:
import subprocess
import sys

def install(packages, no_deps=False):
    cmd = [sys.executable, "-m", "pip", "install"]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(packages)
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)

# OpenVLA compatibility pins
install([
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "peft==0.11.1",
    "accelerate>=0.25.0",
    "sentencepiece==0.1.99",
    "draccus==0.8.0",
    "json-numpy",
    "protobuf",
])

# Standard utilities
install([
    "einops",
    "jsonlines",
    "rich",
    "matplotlib",
    "huggingface_hub",
    "wandb",
])

# Newer bitsandbytes compatible with Kaggle's modern PyTorch/Triton environment
install([
    "bitsandbytes>=0.45.0",
])

# TensorFlow Datasets (we do NOT install a pinned tensorflow — Kaggle's native TF stays)
install([
    "tensorflow_datasets==4.9.3",
])

# DLIMP dataloader — not a PyPI package, installed from OpenVLA's fork.
# --no-deps prevents it from disturbing Kaggle's existing TensorFlow/PyTorch stack.
install([
    "git+https://github.com/moojink/dlimp_openvla"
], no_deps=True)

print("\n✅ All installations completed successfully!")


Running: /usr/bin/python3 -m pip install transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 peft==0.11.1 accelerate>=0.25.0 sentencepiece==0.1.99 draccus==0.8.0 json-numpy protobuf
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 40.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.1 MB/s eta 0:00:00
  Cr

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.


Running: /usr/bin/python3 -m pip install einops jsonlines rich matplotlib huggingface_hub wandb
Running: /usr/bin/python3 -m pip install bitsandbytes>=0.45.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.9 MB/s eta 0:00:00
Running: /usr/bin/python3 -m pip install tensorflow_datasets==4.9.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 41.5 MB/s eta 0:00:00
  Attempting uninstall: tensorflow_datasets
    Found existing installation: tensorflow-datasets 4.9.9
    Uninstalling tensorflow-datasets-4.9.9:
      Successfully uninstalled tensorflow-datasets-4.9.9
Running: /usr/bin/python3 -m pip install --no-deps git+https://github.com/moojink/dlimp_openvla
  Cloning https://github.com/moojink/dlimp_openvla to /tmp/pip-req-build-sow6lrzl


  Running command git clone --filter=blob:none --quiet https://github.com/moojink/dlimp_openvla /tmp/pip-req-build-sow6lrzl


  Resolved https://github.com/moojink/dlimp_openvla to commit 040105d256bd28866cc6620621a3d5f7b6b91b46
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for dlimp: filename=dlimp-0.0.1-py3-none-any.whl size=13525 sha256=5bf91a6d814235fbdfb1e8a3893cdf6c60f9def5ad39eb7bc348cb0f863da547
  Stored in directory: /tmp/pip-ephem-wheel-cache-91fau03z/wheels/82/c2/c8/2fbea45c49caacd75f3528015c28c64546690c3c310446270d
Successfully built dlimp

✅ All installations completed successfully!


## 2. Configure paths and install OpenVLA (editable, no-deps)
(Unchanged.)

In [2]:
import os
import subprocess
import sys

WORKDIR = "/kaggle/working"
OPENVLA_DIR = os.path.join(WORKDIR, "openvla")
DATA_DIR = os.path.join(WORKDIR, "datasets", "openvla", "modified_libero_rlds")
CHECKPOINT_DIR = os.path.join(WORKDIR, "openvla_checkpoints", "smoke_test_run")

if not os.path.exists(OPENVLA_DIR):
    print("Cloning OpenVLA...")
    subprocess.run(["git", "clone", "https://github.com/openvla/openvla", OPENVLA_DIR], check=True)
else:
    print("✅ OpenVLA already cloned.")

# Install OpenVLA natively into the Python environment (editable, no-deps —
# dependencies are already handled explicitly in the cell above)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", OPENVLA_DIR, "--no-deps"], check=True)
print("✅ Installed OpenVLA package.")


Cloning OpenVLA...


Cloning into '/kaggle/working/openvla'...


Obtaining file:///kaggle/working/openvla
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for openvla (pyproject.toml): started
  Building editable for openvla (pyproject.toml): finished with status 'done'
  Created wheel for openvla: filename=openvla-0.0.3-0.editable-py3-none-any.whl size=17763 sha256=16e9ee8046fc58f7867379d001f387428269257bad5691978d0659fc8e3e1c6e
  Stored in directory: /tmp/pip-ephem-wheel-cache-st5n9_53/wheels/b1/ad/f4/d8e4a4b255e02b29eafaade736c6b4f5c56118f2ca7c4fb67b
Successfully built openvla
✅ Installed

## 3. Patch `droid_utils.py` — remove the `tensorflow_graphics` import
(Unchanged — this was needed because `tensorflow_graphics` is only used for the DROID dataset, not LIBERO, but was imported unconditionally.)

In [3]:
patch_file = os.path.join(
    OPENVLA_DIR, "prismatic", "vla", "datasets", "rlds", "oxe", "utils", "droid_utils.py"
)
with open(patch_file, "r") as f:
    content = f.read()

target = "import tensorflow_graphics.geometry.transformation as tfg"
if target in content:
    content = content.replace(target, f"# {target}")
    with open(patch_file, "w") as f:
        f.write(content)
    print("\u2705 Patched droid_utils.py to remove tensorflow_graphics dependency.")
else:
    print("\u2139\ufe0f Exact import line not found verbatim \u2014 checking current state below.")

with open(patch_file, "r") as f:
    verify = f.read()

# Check for a LIVE (uncommented) import specifically \u2014 a naive substring check
# for "import tensorflow_graphics" fails even on a correctly-commented line, since
# "# import tensorflow_graphics..." still contains that substring.
live_imports = [
    line for line in verify.splitlines()
    if line.strip().startswith("import tensorflow_graphics")
]
assert not live_imports, (
    f"\u274c droid_utils.py still has a live (uncommented) tensorflow_graphics import: "
    f"{live_imports}. The patch did not apply \u2014 inspect the file manually before proceeding."
)
print("\u2705 Verified: no live tensorflow_graphics import remains in droid_utils.py.")


✅ Patched droid_utils.py to remove tensorflow_graphics dependency.
✅ Verified: no live tensorflow_graphics import remains in droid_utils.py.


## 4. Patch `finetune.py` — use fp16 instead of bf16 for T4 compatibility
(Unchanged — T4 GPUs have no native bf16 tensor-core support.)

In [4]:
finetune_file = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune.py")
with open(finetune_file, "r") as f:
    ft_content = f.read()

n_torch_dtype = ft_content.count("torch_dtype=torch.bfloat16")
n_bnb = ft_content.count("bnb_4bit_compute_dtype=torch.bfloat16")
n_dtype = ft_content.count("dtype=torch.bfloat16")
n_to = ft_content.count(".to(torch.bfloat16)")
print(f"Occurrences before patch — torch_dtype: {n_torch_dtype}, bnb_4bit_compute_dtype: {n_bnb}, "
      f"dtype=: {n_dtype}, .to(): {n_to}")
if n_torch_dtype > 1:
    print("⚠️ Multiple torch_dtype=torch.bfloat16 occurrences found — patching ALL of them, not just the first.")

# Patch every occurrence, not just the first
ft_content = ft_content.replace("bnb_4bit_compute_dtype=torch.bfloat16", "bnb_4bit_compute_dtype=torch.float16")
ft_content = ft_content.replace("torch_dtype=torch.bfloat16", "torch_dtype=torch.float16")
ft_content = ft_content.replace("dtype=torch.bfloat16", "dtype=torch.float16")
ft_content = ft_content.replace(".to(torch.bfloat16)", ".to(torch.float16)")

with open(finetune_file, "w") as f:
    f.write(ft_content)

assert "torch.bfloat16" not in ft_content, (
    "❌ A bfloat16 reference survived the patch — inspect finetune.py manually for a pattern "
    "this cell did not account for."
)
print("✅ Verified: no remaining bfloat16 references in finetune.py.")


Occurrences before patch — torch_dtype: 2, bnb_4bit_compute_dtype: 1, dtype=: 4, .to(): 1
⚠️ Multiple torch_dtype=torch.bfloat16 occurrences found — patching ALL of them, not just the first.
✅ Verified: no remaining bfloat16 references in finetune.py.


## 5. Preflight import + version + hardware audit
(Unchanged.)

In [5]:
import sys
from pathlib import Path
import torch
import transformers
import peft
import timm
import tensorflow as tf
import tensorflow_datasets as tfds
import bitsandbytes as bnb
import accelerate
from dlimp import DLataset

OPENVLA_DIR_PATH = Path(OPENVLA_DIR)
if str(OPENVLA_DIR_PATH) not in sys.path:
    sys.path.insert(0, str(OPENVLA_DIR_PATH))

prismatic_dir = OPENVLA_DIR_PATH / "prismatic"
print("OpenVLA directory:", OPENVLA_DIR_PATH)
print("OpenVLA exists:", OPENVLA_DIR_PATH.exists())
print("Prismatic directory:", prismatic_dir)
print("Prismatic exists:", prismatic_dir.exists())

if not prismatic_dir.exists():
    raise RuntimeError("❌ The OpenVLA repository does not contain the expected 'prismatic/' directory.")

# Import the specific module we patched, to confirm it imports cleanly post-patch
from prismatic.vla.datasets.rlds.oxe.utils import droid_utils

print("\n--- VERSION AUDIT ---")
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"timm:         {timm.__version__}")
print(f"BitsAndBytes: {bnb.__version__}")
print(f"Accelerate:  {accelerate.__version__}")
print(f"TensorFlow:   {tf.__version__}")
print(f"TFDS:         {tfds.__version__}")

print("\n--- COMPATIBILITY CHECKS ---")
print("✅ DLIMP dataset structures imported successfully.")
print("✅ Prismatic/OpenVLA source imported successfully (post-patch).")

print("\n--- HARDWARE AUDIT ---")
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"GPUs Detected: {gpu_count}")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f" - GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1024**3:.2f} GB)")
else:
    raise RuntimeError("❌ No GPU found! Enable a GPU accelerator in Kaggle notebook settings.")

print("\n✅ VERSION / COMPATIBILITY / HARDWARE AUDIT PASSED.")


OpenVLA directory: /kaggle/working/openvla
OpenVLA exists: True
Prismatic directory: /kaggle/working/openvla/prismatic
Prismatic exists: True

--- VERSION AUDIT ---
Python:       3.12.13
PyTorch:      2.10.0+cu128 (CUDA: 12.8)
Transformers: 4.40.1
PEFT:         0.11.1
timm:         0.9.10
BitsAndBytes: 0.50.2
Accelerate:  1.13.0
TensorFlow:   2.20.0
TFDS:         4.9.3

--- COMPATIBILITY CHECKS ---
✅ DLIMP dataset structures imported successfully.
✅ Prismatic/OpenVLA source imported successfully (post-patch).

--- HARDWARE AUDIT ---
GPUs Detected: 2
 - GPU 0: Tesla T4 (14.56 GB)
 - GPU 1: Tesla T4 (14.56 GB)

✅ VERSION / COMPATIBILITY / HARDWARE AUDIT PASSED.


## 6. Download the LIBERO-Object RLDS dataset (~2.6GB subset only)
(Unchanged.)

In [6]:
os.makedirs(DATA_DIR, exist_ok=True)
print("Downloading LIBERO-Object dataset subset...")
subprocess.run([
    "huggingface-cli", "download", "openvla/modified_libero_rlds",
    "--repo-type", "dataset",
    "--include", "libero_object_no_noops/*",
    "--local-dir", DATA_DIR
], check=True)

if os.path.exists(os.path.join(DATA_DIR, "libero_object_no_noops")):
    print("✅ Dataset verified!")
else:
    raise RuntimeError("❌ Dataset download failed or path is incorrect!")


⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.


Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]Downloading 'libero_object_no_noops/1.0.0/libero_object-train.tfrecord-00003-of-00032' to '/kaggle/working/datasets/openvla/modified_libero_rlds/.cache/huggingface/download/libero_object_no_noops/1.0.0/4SuIvF3-AUI4k8XBbb6E41EWRPg=.c7ebe9e12eb73ad8cfd6702231e3e89fa9e7ed72f609433f98cf70e6966d4661.incomplete'
Download complete. Moving file to /kaggle/working/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/dataset_info.json
Fetching 34 files:   3%|▎         | 1/34 [00:00<00:06,  5.45it/s]Downloading 'libero_object_no_noops/1.0.0/features.json' to '/kaggle/working/datasets/openvla/modified_libero_rlds/.cache/huggingface/download/libero_object_no_noops/1.0.0/lrjysyCGup8vrOWNy55Skwbctlw=.a5c8abcf90cb16a51b2e7900f4bdbdafa0566816.incomplete'
Download complete. Moving file to /kaggle/working/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/features.json
Download complete. Moving file to /kaggle/working/

/kaggle/working/datasets/openvla/modified_libero_rlds
✅ Dataset verified!


Download complete. Moving file to /kaggle/working/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/libero_object-train.tfrecord-00025-of-00032
Fetching 34 files: 100%|██████████| 34/34 [00:09<00:00,  3.61it/s]


## 7. RUN CONFIG — the only real changes from the smoke test

Everything that makes this a "real baseline run" instead of a smoke test lives in this one cell. Nothing below this cell needs editing to change the run length — change these numbers only.

- **MAX_STEPS**: total training steps. The smoke test's own log showed ~1.72s/step once training   was underway (11 steps in 24s), plus a fixed ~250s of one-time overhead (checkpoint download/load,   DDP setup). At 2000 steps: ~2000 × 1.72s ≈ 57 min, + ~4 min overhead ≈ **about 1 hour total**.   Adjust based on your actual Kaggle GPU-hour budget before the deadline.
- **SAVE_STEPS**: how often a checkpoint is written. Set well below MAX_STEPS but not so low that   frequent saves eat meaningful time (each save writes the LoRA adapter + optimizer state).
- **RUN_NAME**: labels the checkpoint directory so this baseline run's output never collides with   or overwrites the smoke test's `smoke_test_run` output.
- Batch size and quantization are deliberately **kept identical to the smoke test**   (`batch_size=1`, `use_quantization=True`) rather than increased, because that configuration is the   only one that's actually been proven to fit in a 16GB T4's memory for this model. Increasing batch   size now, unverified, risks reopening the exact OOM/memory problems this whole pipeline was built   to avoid. If you want a larger *effective* batch size without more memory, increase   `GRAD_ACCUMULATION_STEPS` instead (accumulates gradients over N steps before each optimizer update   — costs more wall-clock time, not more memory).

## ⚠️ UPDATE after a real failed run: mid-training OOM (SIGKILL)

A real 2000-step run confirmed with hard evidence (exact timestamps from the training log):
- The checkpoint save at **step 500** completed cleanly (started 1309.7s, finished 166s later).
- The checkpoint save at **step 1000** was killed by the OS (`SIGKILL`, exit code -9) **83.5s into
  the exact same reload procedure** that had just worked fine 500 steps earlier.

Same code path succeeding once and then failing on identical work later is the signature of
**cumulative RAM growth over the course of training**, not a hard memory ceiling hit immediately.
Each periodic checkpoint save briefly reloads the full ~15GB base checkpoint on top of whatever RAM
is already resident — and by step 1000, that "whatever" had grown enough to tip the process over.

The prime suspect: `--shuffle_buffer_size` defaults to **100,000** in `FinetuneConfig`, and its own
docstring says *"can reduce if OOM"* — a strong, already-documented signal this is exactly the kind
of failure it's meant to address. A buffer that size, holding decoded image trajectories over hundreds
of steps, is a very plausible source of steadily climbing RAM. Two changes below address this:
lowering `SHUFFLE_BUFFER_SIZE` (the root-cause fix) and raising `SAVE_STEPS` (reduces how many times
the risky reload happens at all, as a second layer of safety).

In [7]:
# ============================================================
# RUN CONFIGURATION — edit these, nothing else, to change the run
# ============================================================
MAX_STEPS = 2000              # total training steps for this baseline run
SAVE_STEPS = 2000              # RAISED from 500: a real run showed cumulative RAM growth made
                                # the periodic checkpoint reload progressively riskier; saving
                                # less often means fewer chances to hit that reload spike
GRAD_ACCUMULATION_STEPS = 1    # increase this (not batch_size) for a larger effective batch
SHUFFLE_BUFFER_SIZE = 10     # LOWERED from FinetuneConfig's default of 100,000 -- its own
                                # docstring says "can reduce if OOM", and a real run's timing
                                # (step-500 save succeeded, step-1000 save was SIGKILLed 83.5s
                                # into the identical reload) points at cumulative RAM growth
                                # from the data pipeline as the likely cause. 1000 is generous
                                # for LIBERO-Object's small size; lower further if OOM recurs.
RUN_NAME = "baseline_uniform_lora_run"   # distinct from the smoke test's "smoke_test_run"

assert SAVE_STEPS <= MAX_STEPS, "SAVE_STEPS must not exceed MAX_STEPS"

# Re-derive the checkpoint directory using RUN_NAME instead of the smoke test's
# hardcoded "smoke_test_run" — WORKDIR/OPENVLA_DIR/DATA_DIR come from cell 2 above,
# unchanged.
import os
CHECKPOINT_DIR = os.path.join(WORKDIR, "openvla_checkpoints", RUN_NAME)

est_seconds = MAX_STEPS * 1.72 + 250  # ~1.72s/step observed in the smoke test, + fixed overhead
print(f"Run name:        {RUN_NAME}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print(f"MAX_STEPS:            {MAX_STEPS}")
print(f"SAVE_STEPS:           {SAVE_STEPS}")
print(f"SHUFFLE_BUFFER_SIZE:  {SHUFFLE_BUFFER_SIZE}")
print(f"Rough estimated runtime: {est_seconds/60:.1f} minutes "
      f"(based on the smoke test's observed ~1.72s/step — actual time may vary)")


Run name:        baseline_uniform_lora_run
Checkpoint dir:  /kaggle/working/openvla_checkpoints/baseline_uniform_lora_run
MAX_STEPS:            2000
SAVE_STEPS:           2000
SHUFFLE_BUFFER_SIZE:  10
Rough estimated runtime: 61.5 minutes (based on the smoke test's observed ~1.72s/step — actual time may vary)


## 8. Patch DDP + Accelerate

Two fixes proven necessary by real Kaggle failures, both unchanged in substance from the working smoke test — consolidated here into ONE clean application each (the smoke-test notebook applied the DDP fix twice in a row by accident, which was harmless but redundant; this version applies it once).

**Fix 1 — DDP CPU-tensor crash.** PEFT's newly-attached LoRA adapter weights are created on CPU by default, while the 4-bit quantized base model stays on GPU. DDP's NCCL backend can't broadcast CPU tensors during its state-sync step, causing `RuntimeError: No backend type associated with device type cpu`. Fix: manually move any CPU-resident params/buffers to `device_id` before wrapping in `DDP(...)`, and set `_set_static_graph()` to avoid a related "double-ready" DDP crash.

**Fix 2 — Accelerate `.to()` on a quantized model.** `accelerate`'s `dispatch_model()` unconditionally calls `model.to(device)` in its single-GPU branch. `transformers` overrides `.to()` to raise on any 4-bit/8-bit model. Traced through `transformers` 4.40.1's actual source: the reliable signal that a model is quantized *before* `dispatch_model()` runs is `model.is_quantized` (set earlier via `hf_quantizer.preprocess_model()`) — NOT `is_loaded_in_4bit`/`is_loaded_in_8bit`, which are only set *after* `dispatch_model()` runs and so are always `False` at the point this check needs them. Fix: skip `.to()` when `model.is_quantized` is `True`.

In [8]:
import os
import re
import shutil
import sys
import subprocess
import accelerate

finetune_file = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune.py")

# ------------------------------------------------------------------
# Fix 1: move any CPU-resident LoRA params/buffers to GPU before DDP,
# and set static graph. Applied ONCE (the smoke test applied this twice
# by accident when two cells matched the same target string in sequence;
# harmless there since the second pass found nothing left to move, but
# cleaner to do it once here).
# ------------------------------------------------------------------
with open(finetune_file, "r") as f:
    ft_content = f.read()

target_ddp = "vla = DDP(vla, device_ids=[device_id], find_unused_parameters=True, gradient_as_bucket_view=True)"

device_move_code = """
    # Manually move any CPU-resident LoRA weights/buffers to the GPU before DDP.
    # Needed because PEFT's newly-attached LoRA adapter params are created on CPU
    # by default, while the 4-bit quantized base model already lives on GPU — and
    # DDP's NCCL backend cannot broadcast CPU tensors during state sync.
    for param in vla.parameters():
        if param.device.type == 'cpu':
            param.data = param.data.to(device_id)
    for buffer in vla.buffers():
        if buffer.device.type == 'cpu':
            buffer.data = buffer.data.to(device_id)

    vla = DDP(vla, device_ids=[device_id], find_unused_parameters=True, gradient_as_bucket_view=True)
    vla._set_static_graph()  # avoids a related "double-ready" DDP crash
"""

if device_move_code.strip() in ft_content:
    print("✅ finetune.py already patched for CPU->GPU param move + static graph.")
elif target_ddp in ft_content:
    ft_content = ft_content.replace(target_ddp, device_move_code)
    with open(finetune_file, "w") as f:
        f.write(ft_content)
    print("✅ Patched finetune.py: CPU-resident params moved to GPU before DDP, static graph set.")
else:
    raise RuntimeError(
        "Could not find the expected DDP initialization line in finetune.py. "
        "This patch is CONFIRMED necessary (a real Kaggle run failed here with "
        "'RuntimeError: No backend type associated with device type cpu') \u2014 "
        "inspect finetune.py's finetune() function manually for the current DDP(...) call."
    )

# ------------------------------------------------------------------
# Fix 2: patch accelerate's dispatch_model() so it doesn't call .to()
# on an already-quantized model.
# ------------------------------------------------------------------
accelerate_dir = os.path.dirname(accelerate.__file__)
accelerate_file = os.path.join(accelerate_dir, "big_modeling.py")
print(f"Accelerate big_modeling.py: {accelerate_file}")

if not os.path.exists(accelerate_file):
    raise FileNotFoundError(f"Could not find Accelerate big_modeling.py at:\n{accelerate_file}")

with open(accelerate_file, "r") as f:
    source = f.read()

# Confirmed via real Kaggle traceback:
#   transformers/modeling_utils.py -> from_pretrained -> dispatch_model(...)
#   -> accelerate/big_modeling.py: "model.to(device)"
#   -> transformers raises: "ValueError: `.to` is not supported for `4-bit` or
#      `8-bit` bitsandbytes models."
# is_loaded_in_4bit / is_loaded_in_8bit are set by hf_quantizer.postprocess_model(),
# which runs AFTER dispatch_model() — so checking those here never blocks the call.
# model.is_quantized is set BEFORE dispatch_model() (via hf_quantizer.preprocess_model())
# and is the reliable signal, confirmed by reading transformers 4.40.1's actual source.
old_snippet = "        if device != \"disk\":\n            model.to(device)\n"
new_snippet = (
    "        if device != \"disk\":\n"
    "            if not getattr(model, \"is_quantized\", False):\n"
    "                model.to(device)\n"
)

patched = False
if new_snippet in source:
    print("✅ Accelerate is already patched.")
    patched = True
elif old_snippet in source:
    source = source.replace(old_snippet, new_snippet, 1)
    with open(accelerate_file, "w") as f:
        f.write(source)
    print("✅ Patched Accelerate's dispatch_model() single-device branch (exact match).")
    patched = True
else:
    pattern = re.compile(r'(\n(\s*)if device != "disk":\n)(\s*)model\.to\(device\)\n')
    def _sub(m):
        line_indent, body_indent = m.group(2), m.group(3)
        return (
            f'{m.group(1)}{body_indent}if not getattr(model, "is_quantized", False):\n'
            f'{body_indent}    model.to(device)\n'
        )
    new_source, n = pattern.subn(_sub, source, count=1)
    if n == 1:
        with open(accelerate_file, "w") as f:
            f.write(new_source)
        print("✅ Patched Accelerate's dispatch_model() single-device branch (regex fallback).")
        patched = True

if not patched:
    raise RuntimeError(
        "Could not patch Accelerate's dispatch_model() to skip .to() on quantized models "
        f"(installed Accelerate: {accelerate.__version__}, file: {accelerate_file}). "
        "This IS confirmed necessary \u2014 inspect big_modeling.py's dispatch_model() manually."
    )

print("\n✅ All patches applied. Ready to launch training.")


✅ Patched finetune.py: CPU-resident params moved to GPU before DDP, static graph set.
Accelerate big_modeling.py: /usr/local/lib/python3.12/dist-packages/accelerate/big_modeling.py
✅ Patched Accelerate's dispatch_model() single-device branch (exact match).

✅ All patches applied. Ready to launch training.


## 9. Launch the baseline training run

Same command shape as the proven smoke test, with `--max_steps`/`--save_steps` now pulled from the config cell above instead of hardcoded, and the run/checkpoint directory set to `RUN_NAME` so it never collides with the smoke test's output. `batch_size=1` and `use_quantization=True` are unchanged from the smoke test on purpose (see the reasoning in the config cell above).

In [9]:
os.environ["WANDB_MODE"] = "disabled"

if os.path.exists(CHECKPOINT_DIR):
    print(f"⚠️ {CHECKPOINT_DIR} already exists from a previous attempt — removing it "
          "to start this run clean.")
    shutil.rmtree(CHECKPOINT_DIR)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
adapter_tmp_dir = os.path.join(CHECKPOINT_DIR, "tmp")

finetune_script = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune.py")
if not os.path.exists(finetune_script):
    raise FileNotFoundError(f"OpenVLA finetune.py not found:\n{finetune_script}")

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", "--nnodes=1", "--nproc-per-node=1",
    finetune_script,
    "--vla_path", "openvla/openvla-7b",
    "--data_root_dir", DATA_DIR,
    "--dataset_name", "libero_object_no_noops",
    "--run_root_dir", CHECKPOINT_DIR,
    "--adapter_tmp_dir", adapter_tmp_dir,
    "--use_lora", "True",
    "--use_quantization", "True",     # unchanged from the smoke test — proven to fit 16GB T4
    "--lora_rank", "32",              # this is the parameter budget the MMD-selective run must match
    "--batch_size", "1",              # unchanged from the smoke test — see config cell reasoning
    "--shuffle_buffer_size", str(SHUFFLE_BUFFER_SIZE),  # lowered to fight cumulative RAM growth
    "--grad_accumulation_steps", str(GRAD_ACCUMULATION_STEPS),
    "--learning_rate", "5e-4",
    "--image_aug", "False",
    "--wandb_project", "",
    "--wandb_entity", "",
    "--save_steps", str(SAVE_STEPS),
    "--max_steps", str(MAX_STEPS),
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"   # force single-GPU to match --nproc-per-node=1
env["WANDB_MODE"] = "disabled"

print("\n" + "=" * 70)
print(f"STARTING BASELINE RUN: {RUN_NAME}")
print("=" * 70)
print(f"Python: {sys.executable}")
print(f"GPU: CUDA_VISIBLE_DEVICES={env['CUDA_VISIBLE_DEVICES']}")
print(f"Training script: {finetune_script}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"MAX_STEPS={MAX_STEPS}  SAVE_STEPS={SAVE_STEPS}  GRAD_ACCUMULATION_STEPS={GRAD_ACCUMULATION_STEPS}")
print("\nCommand:")
print(" ".join(cmd))
print("=" * 70 + "\n")

log_path = os.path.join(CHECKPOINT_DIR, "training.log")
with open(log_path, "w") as log_file:
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd=OPENVLA_DIR, env=env,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    process.wait()

print("\n" + "=" * 70)
if process.returncode != 0:
    print(f"❌ Training failed with exit code {process.returncode}")
    print(f"Full log saved to: {log_path}")
    raise RuntimeError(f"Baseline run failed with exit code {process.returncode}")

print(f"✅ Training process exited successfully! ({MAX_STEPS} steps)")
print("=" * 70)



STARTING BASELINE RUN: baseline_uniform_lora_run
Python: /usr/bin/python3
GPU: CUDA_VISIBLE_DEVICES=0
Training script: /kaggle/working/openvla/vla-scripts/finetune.py
Checkpoint directory: /kaggle/working/openvla_checkpoints/baseline_uniform_lora_run
MAX_STEPS=2000  SAVE_STEPS=2000  GRAD_ACCUMULATION_STEPS=1

Command:
/usr/bin/python3 -m torch.distributed.run --standalone --nnodes=1 --nproc-per-node=1 /kaggle/working/openvla/vla-scripts/finetune.py --vla_path openvla/openvla-7b --data_root_dir /kaggle/working/datasets/openvla/modified_libero_rlds --dataset_name libero_object_no_noops --run_root_dir /kaggle/working/openvla_checkpoints/baseline_uniform_lora_run --adapter_tmp_dir /kaggle/working/openvla_checkpoints/baseline_uniform_lora_run/tmp --use_lora True --use_quantization True --lora_rank 32 --batch_size 1 --shuffle_buffer_size 10 --grad_accumulation_steps 1 --learning_rate 5e-4 --image_aug False --wandb_project  --wandb_entity  --save_steps 2000 --max_steps 2000

[W911 00:26:07.6

## 10. End-to-end verification

Same checks as the smoke test, generalized to use `MAX_STEPS` instead of a hardcoded "Step 10" — this is what actually confirms the run is a valid, usable baseline result and not just a process that exited without crashing.

In [10]:
import glob
import json

with open(os.path.join(CHECKPOINT_DIR, "training.log"), "r") as f:
    logs = f.read()

step_marker_found = (
    f"Step {MAX_STEPS}" in logs
    or f"step {MAX_STEPS}" in logs
    or f"{MAX_STEPS}/{MAX_STEPS}" in logs
    or f"[{MAX_STEPS}/{MAX_STEPS}]" in logs
    or f"Max step {MAX_STEPS} reached" in logs
)
trainable_found = "trainable params" in logs

assert step_marker_found, f"❌ Training did not demonstrably reach step {MAX_STEPS}."
assert trainable_found, "❌ Could not verify trainable LoRA parameters."
print(f"✅ Log verification: training reached step {MAX_STEPS}, PEFT attached.")

adapters = glob.glob(os.path.join(CHECKPOINT_DIR, "**", "adapter_model.safetensors"), recursive=True)
configs = glob.glob(os.path.join(CHECKPOINT_DIR, "**", "adapter_config.json"), recursive=True)

assert adapters, "❌ No LoRA adapter was saved."
assert configs, "❌ No adapter_config.json was saved."

adapter_path = adapters[0]
config_path = configs[0]
print(f"\n✅ Found LoRA adapter at: {adapter_path}")

size_mb = os.path.getsize(adapter_path) / (1024 * 1024)
print(f"Adapter size: {size_mb:.2f} MB")

with open(config_path, "r") as f:
    cfg = json.load(f)
    print(f"LoRA rank (r): {cfg.get('r')}")
    print(f"Target modules: {cfg.get('target_modules')}")

assert cfg.get("r") == 32, f"❌ Expected LoRA rank 32, got {cfg.get('r')}"
assert size_mb > 1.0, "❌ Adapter suspiciously small (<1MB) — target modules likely failed."

from peft import PeftConfig
peft_cfg = PeftConfig.from_pretrained(os.path.dirname(config_path))
assert peft_cfg.r == 32, "❌ Reloaded PEFT config rank does not match."
assert peft_cfg.target_modules, "❌ Reloaded PEFT config has no target modules."
print("✅ PEFT adapter configuration is valid and loadable.")

print(f"\n🚀 BASELINE RUN COMPLETE: {RUN_NAME}, {MAX_STEPS} steps, checkpoint at {CHECKPOINT_DIR}")
print("This checkpoint is the uniform-LoRA baseline for the paper comparison.")
print("Next: build and run the MMD-selective LoRA condition at the SAME lora_rank=32 budget,")
print("then evaluate BOTH checkpoints with LIBERO-Object rollout success rate.")


✅ Log verification: training reached step 2000, PEFT attached.

✅ Found LoRA adapter at: /kaggle/working/openvla_checkpoints/baseline_uniform_lora_run/tmp/openvla-7b+libero_object_no_noops+b1+lr-0.0005+lora-r32+dropout-0.0+q-4bit/adapter_model.safetensors
Adapter size: 923.90 MB
LoRA rank (r): 32
Target modules: ['q', 'q_proj', 'lm_head', 'fc3', 'fc1', 'kv', 'o_proj', 'gate_proj', 'proj', 'v_proj', 'up_proj', 'qkv', 'k_proj', 'down_proj', 'fc2']
✅ PEFT adapter configuration is valid and loadable.

🚀 BASELINE RUN COMPLETE: baseline_uniform_lora_run, 2000 steps, checkpoint at /kaggle/working/openvla_checkpoints/baseline_uniform_lora_run
This checkpoint is the uniform-LoRA baseline for the paper comparison.
Next: build and run the MMD-selective LoRA condition at the SAME lora_rank=32 budget,
then evaluate BOTH checkpoints with LIBERO-Object rollout success rate.
